In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("EV_Charging_Training") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

# ============================================================================
# LOAD FROM SILVER LAYER
# ============================================================================
print("="*80)
print("LOADING PREPROCESSED DATA FROM SILVER")
print("="*80)

silver_path = "hdfs://localhost:9000/ev-project/data/silver/ev_sessions_preprocessed"

# Đọc dữ liệu đã preprocess
df_silver = spark.read.parquet(silver_path)
print(f"✓ Loaded {df_silver.count():,} rows from Silver")

# Drop remaining NULLs (if any)
df_silver = df_silver.dropna()
print(f"✓ After dropping NULLs: {df_silver.count():,} rows")

# ============================================================================
# TRAIN/TEST SPLIT (Temporal order, no shuffle)
# ============================================================================
print("\n" + "="*80)
print("TRAIN/TEST SPLIT (80/20 - Temporal Order)")
print("="*80)

from pyspark.sql.window import Window

# Sort by time
df_sorted = df_silver.orderBy("connectionTime")

# Add row number
df_with_row = df_sorted.rdd.zipWithIndex().toDF()
df_with_row = df_with_row.withColumnRenamed("_1", "data").withColumn("row_id", col("_2"))

total_rows = df_sorted.count()
split_point = int(total_rows * 0.8)

print(f"Total rows: {total_rows:,}")
print(f"Train rows: {split_point:,} (80%)")
print(f"Test rows: {total_rows - split_point:,} (20%)")

# Split
train_df = df_with_row.filter(col("row_id") < split_point).select("data.*")
test_df = df_with_row.filter(col("row_id") >= split_point).select("data.*")

# Drop row_id column
train_df = train_df.drop("row_id")
test_df = test_df.drop("row_id")

print(f"\n✓ Train set: {train_df.count():,} rows")
print(f"✓ Test set: {test_df.count():,} rows")

# ============================================================================
# PREPARE FEATURES FOR MODELING
# ============================================================================
print("\n" + "="*80)
print("PREPARING FEATURES FOR MODELING")
print("="*80)

# Feature columns (same as paper)
feature_cols = [
    "hour", "day_of_week", "month", "season",
    "duration", "charging_duration", "charging_duration_log",
    "hour_sin", "hour_cos", "day_of_year", "week_of_year",
    "is_holiday",
    "lag_1_log", "lag_2_log", "lag_3_log",
    "rolling_mean_3_log", "rolling_mean_5_log"
]

target_col = "kWhDelivered"

print(f"✓ {len(feature_cols)} features ready")
print(f"✓ Target: {target_col}")

# Vector assemble
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features", 
                        withStd=True, withMean=True)

pipeline = Pipeline(stages=[assembler, scaler])

# Fit on train, transform both
pipeline_model = pipeline.fit(train_df)
train_df = pipeline_model.transform(train_df)
test_df = pipeline_model.transform(test_df)

print("✓ Features assembled and scaled")

# ============================================================================
# TRAIN RANDOM FOREST MODEL
# ============================================================================
print("\n" + "="*80)
print("TRAINING RANDOM FOREST MODEL")
print("="*80)

from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Random Forest with optimized params
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=target_col,
    numTrees=100,
    maxDepth=15,
    minInstancesPerNode=5,
    seed=42,
    featureSubsetStrategy="sqrt"
)

print("✓ Random Forest configured:")
print(f"  - numTrees: 100")
print(f"  - maxDepth: 15")
print(f"  - minInstancesPerNode: 5")

# Train
print("\nTraining model...")
rf_model = rf.fit(train_df)

# Predict
train_pred = rf_model.transform(train_df)
test_pred = rf_model.transform(test_df)

# ============================================================================
# EVALUATION
# ============================================================================
print("\n" + "="*80)
print("MODEL EVALUATION")
print("="*80)

evaluator_rmse = RegressionEvaluator(labelCol=target_col, metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol=target_col, metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol=target_col, metricName="r2")

# Train metrics
train_rmse = evaluator_rmse.evaluate(train_pred)
train_mae = evaluator_mae.evaluate(train_pred)
train_r2 = evaluator_r2.evaluate(train_pred)

# Test metrics
test_rmse = evaluator_rmse.evaluate(test_pred)
test_mae = evaluator_mae.evaluate(test_pred)
test_r2 = evaluator_r2.evaluate(test_pred)

print("\n📊 TRAIN SET:")
print(f"  RMSE: {train_rmse:.4f}")
print(f"  MAE:  {train_mae:.4f}")
print(f"  R²:   {train_r2:.4f}")

print("\n📊 TEST SET:")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  MAE:  {test_mae:.4f}")
print(f"  R²:   {test_r2:.4f}")

# ============================================================================
# FEATURE IMPORTANCE
# ============================================================================
print("\n" + "="*80)
print("FEATURE IMPORTANCE")
print("="*80)

feature_importance = rf_model.featureImportances
features_imp = [(feature_cols[i], float(feature_importance[i])) 
                for i in range(len(feature_cols))]
features_imp.sort(key=lambda x: x[1], reverse=True)

print("\nTop 10 most important features:")
for i, (feature, importance) in enumerate(features_imp[:10], 1):
    print(f"  {i:2d}. {feature:25s}: {importance:.4f}")

# ============================================================================
# SAVE MODEL
# ============================================================================
print("\n" + "="*80)
print("SAVING MODEL")
print("="*80)

model_path = "hdfs://localhost:9000/ev-project/data/gold/models/rf_model"
rf_model.write().overwrite().save(model_path)
print(f"✓ Model saved to: {model_path}")

print("\n" + "="*80)
print("✅ PHASE 2 COMPLETE!")
print("="*80)

LOADING PREPROCESSED DATA FROM SILVER


✓ Loaded 31,393 rows from Silver


✓ After dropping NULLs: 31,391 rows

TRAIN/TEST SPLIT (80/20 - Temporal Order)


Total rows: 31,391
Train rows: 25,112 (80%)
Test rows: 6,279 (20%)



✓ Train set: 25,112 rows


...

26/04/11 14:11:35 WARN DAGScheduler: Broadcasting large task binary with size 1096.8 KiB
26/04/11 14:11:38 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/04/11 14:11:42 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB
26/04/11 14:11:48 WARN DAGScheduler: Broadcasting large task binary with size 1139.0 KiB
26/04/11 14:12:02 WARN DAGScheduler: Broadcasting large task binary with size 6.9 MiB
26/04/11 14:12:06 WARN DAGScheduler: Broadcasting large task binary with size 1985.3 KiB
26/04/11 14:12:09 WARN DAGScheduler: Broadcasting large task binary with size 11.8 MiB
26/04/11 14:12:15 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
26/04/11 14:12:19 WARN DAGScheduler: Broadcasting large task binary with size 18.8 MiB
26/04/11 14:12:26 WARN DAGScheduler: Broadcasting large task binary with size 4.5 MiB
26/04/11 14:12:33 WARN DAGScheduler: Broadcasting large task binary with size 27.8 MiB
26/04/11 14:12:41 WARN DAGScheduler: Broad


MODEL EVALUATION



📊 TRAIN SET:
  RMSE: 3.5083
  MAE:  2.3221
  R²:   0.7022

📊 TEST SET:
  RMSE: 8.2933
  MAE:  5.3916
  R²:   -0.0277

FEATURE IMPORTANCE

Top 10 most important features:
   1. charging_duration        : 0.3027
   2. charging_duration_log    : 0.2995
   3. duration                 : 0.0897
   4. day_of_week              : 0.0331
   5. hour                     : 0.0300
   6. rolling_mean_5_log       : 0.0288
   7. lag_1_log                : 0.0255
   8. lag_2_log                : 0.0248
   9. hour_cos                 : 0.0243
  10. lag_3_log                : 0.0241

SAVING MODEL


26/04/11 14:14:33 WARN TaskSetManager: Stage 258 contains a task of very large size (4669 KiB). The maximum recommended task size is 1000 KiB.
[Stage 260:>                                                        (0 + 1) / 1]

✓ Model saved to: hdfs://localhost:9000/ev-project/data/gold/models/rf_model

✅ PHASE 2 COMPLETE!


In [4]:
# ============================================================================
# RESET SPARK & KHỞI TẠO VỚI CẤU HÌNH 4GB RAM
# ============================================================================
from pyspark import SparkContext
from pyspark.sql import SparkSession

# Stop existing Spark context
try:
    SparkContext._active_spark_context.stop()
except:
    pass

# Create new Spark session
spark = SparkSession.builder \
    .appName("EV_Charging_GBT_Paper") \
    .master("local[2]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.memory.fraction", "0.6") \
    .getOrCreate()

print("="*80)
print("SPARK CONFIGURED FOR 4GB RAM")
print("="*80)

# ============================================================================
# LOAD DATA FROM SILVER
# ============================================================================
print("\n[1/6] Loading preprocessed data...")
silver_path = "hdfs://localhost:9000/ev-project/data/silver/ev_sessions_preprocessed"
df_silver = spark.read.parquet(silver_path)
df_silver = df_silver.dropna()
print(f"✓ Loaded {df_silver.count():,} rows")

# ============================================================================
# TRAIN/TEST SPLIT (80/20, temporal order, no shuffle - as per paper)
# ============================================================================
print("\n[2/6] Train/test split (80/20, temporal order)...")
df_sorted = df_silver.orderBy("connectionTime")
df_with_row = df_sorted.withColumn("row_id", monotonically_increasing_id())

total_rows = df_sorted.count()
split_point = int(total_rows * 0.8)

train_df = df_with_row.filter(col("row_id") < split_point).drop("row_id")
test_df = df_with_row.filter(col("row_id") >= split_point).drop("row_id")

# Repartition to avoid OOM
train_df = train_df.repartition(4)
test_df = test_df.repartition(4)

print(f"✓ Train: {train_df.count():,} rows (80%)")
print(f"✓ Test: {test_df.count():,} rows (20%)")

# ============================================================================
# FEATURE PREPARATION (17 features as per paper)
# ============================================================================
print("\n[3/6] Preparing features...")

feature_cols = [
    "hour", "day_of_week", "month", "season",
    "duration", "charging_duration", "charging_duration_log",
    "hour_sin", "hour_cos", "day_of_year", "week_of_year",
    "is_holiday",
    "lag_1_log", "lag_2_log", "lag_3_log",
    "rolling_mean_3_log", "rolling_mean_5_log"
]

target_col = "kWhDelivered"

from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
pipeline = Pipeline(stages=[assembler])

pipeline_model = pipeline.fit(train_df)
train_df = pipeline_model.transform(train_df)
test_df = pipeline_model.transform(test_df)

print(f"✓ {len(feature_cols)} features prepared")

# ============================================================================
# TRAIN GBT (Gradient-Boosted Trees) - As per paper Table 3
# ============================================================================
print("\n[4/6] Training GBT model...")

from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# GBT hyperparameters based on paper Table 3
gbt = GBTRegressor(
    featuresCol="features",
    labelCol=target_col,
    maxIter=100,              # Number of trees (similar to n_estimators in XGBoost)
    maxDepth=5,               # Tree depth (paper: 3-10 range)
    stepSize=0.1,             # Learning rate
    subsamplingRate=0.8,      # Row sampling
    minInstancesPerNode=5,    # Min samples per leaf
    seed=42,
    lossType="squared"        # Least squares regression
)

print("✓ GBT configured (as per paper Table 3):")
print(f"  - maxIter (trees): {gbt.getMaxIter()}")
print(f"  - maxDepth: {gbt.getMaxDepth()}")
print(f"  - stepSize (learning rate): {gbt.getStepSize()}")
print(f"  - subsamplingRate: {gbt.getSubsamplingRate()}")

# Train
gbt_model = gbt.fit(train_df)
print("✓ Training complete")

# ============================================================================
# PREDICTIONS
# ============================================================================
print("\n[5/6] Making predictions...")
train_pred = gbt_model.transform(train_df)
test_pred = gbt_model.transform(test_df)

# ============================================================================
# EVALUATION (As per paper: MAE, MSE, RMSE, R²)
# ============================================================================
print("\n[6/6] Evaluation...")

evaluator_rmse = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="mae")
evaluator_mse = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="mse")
evaluator_r2 = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="r2")

# Train metrics
train_rmse = evaluator_rmse.evaluate(train_pred)
train_mae = evaluator_mae.evaluate(train_pred)
train_mse = evaluator_mse.evaluate(train_pred)
train_r2 = evaluator_r2.evaluate(train_pred)

# Test metrics
test_rmse = evaluator_rmse.evaluate(test_pred)
test_mae = evaluator_mae.evaluate(test_pred)
test_mse = evaluator_mse.evaluate(test_pred)
test_r2 = evaluator_r2.evaluate(test_pred)

# ============================================================================
# PRINT RESULTS
# ============================================================================
print("\n" + "="*80)
print("GBT MODEL RESULTS (As per paper evaluation metrics)")
print("="*80)

print("\n📊 TRAIN SET:")
print(f"  RMSE: {train_rmse:.4f}")
print(f"  MAE:  {train_mae:.4f}")
print(f"  MSE:  {train_mse:.4f}")
print(f"  R²:   {train_r2:.4f}")

print("\n📊 TEST SET:")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  MAE:  {test_mae:.4f}")
print(f"  MSE:  {test_mse:.4f}")
print(f"  R²:   {test_r2:.4f}")

# ============================================================================
# FEATURE IMPORTANCE (As per paper Fig. 9 - SHAP-like analysis)
# ============================================================================
print("\n" + "="*80)
print("FEATURE IMPORTANCE (GBT - comparable to paper's XGBoost)")
print("="*80)

feature_importance = gbt_model.featureImportances
features_imp = [(feature_cols[i], float(feature_importance[i])) 
                for i in range(len(feature_cols))]
features_imp.sort(key=lambda x: x[1], reverse=True)

print("\nTop 10 most important features (as per paper Fig. 9):")
for i, (feature, importance) in enumerate(features_imp[:10], 1):
    bar = "█" * int(importance * 50)
    print(f"  {i:2d}. {feature:25s}: {importance:.4f} {bar}")

# ============================================================================
# COMPARE WITH PAPER'S RESULTS
# ============================================================================
print("\n" + "="*80)
print("COMPARISON WITH PAPER'S RESULTS")
print("="*80)

paper_results = {
    "XGBoost": {"RMSE": 2.31, "MAE": 1.58, "R2": 0.71},
    "LightGBM": {"RMSE": 2.33, "MAE": 1.60, "R2": 0.70},
    "Hybrid_3": {"RMSE": 2.24, "MAE": 1.53, "R2": 0.73}
}

print(f"\n{'Model':<12} {'Our GBT Test':<15} {'Paper XGBoost':<15} {'Paper Hybrid 3':<15}")
print("-" * 60)
print(f"{'RMSE':<12} {test_rmse:<15.4f} {paper_results['XGBoost']['RMSE']:<15.2f} {paper_results['Hybrid_3']['RMSE']:<15.2f}")
print(f"{'MAE':<12} {test_mae:<15.4f} {paper_results['XGBoost']['MAE']:<15.2f} {paper_results['Hybrid_3']['MAE']:<15.2f}")
print(f"{'R²':<12} {test_r2:<15.4f} {paper_results['XGBoost']['R2']:<15.2f} {paper_results['Hybrid_3']['R2']:<15.2f}")

# ============================================================================
# SAVE MODEL
# ============================================================================
print("\n" + "="*80)
print("SAVING MODEL")
print("="*80)

model_path = "hdfs://localhost:9000/ev-project/data/gold/models/gbt_model"
gbt_model.write().overwrite().save(model_path)
print(f"✓ Model saved to: {model_path}")

# ============================================================================
# SAVE PREDICTIONS FOR ANALYSIS
# ============================================================================
print("\n" + "="*80)
print("SAVING PREDICTIONS")
print("="*80)

# Save test predictions with actual values for residual analysis (as per paper Fig. 8)
test_pred.select("kWhDelivered", "prediction") \
    .withColumn("residual", col("kWhDelivered") - col("prediction")) \
    .write.mode("overwrite") \
    .parquet("hdfs://localhost:9000/ev-project/data/gold/predictions/gbt_predictions")

print("✓ Predictions saved for residual analysis")

print("\n" + "="*80)
print("✅ GBT MODEL TRAINING COMPLETE!")
print("="*80)

26/04/11 15:43:09 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


SPARK CONFIGURED FOR 4GB RAM

[1/6] Loading preprocessed data...


✓ Loaded 31,391 rows

[2/6] Train/test split (80/20, temporal order)...


✓ Train: 25,112 rows (80%)


✓ Test: 6,279 rows (20%)

[3/6] Preparing features...


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


✓ 17 features prepared

[4/6] Training GBT model...
✓ GBT configured (as per paper Table 3):
  - maxIter (trees): 100
  - maxDepth: 5
  - stepSize (learning rate): 0.1
  - subsamplingRate: 0.8


KeyboardInterrupt: 